In [1]:
# %load_ext autoreload
# %autoreload 2

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os
from pathlib import Path
from typing import Tuple, List, Set

# Add project root to sys.path
project_root = Path('../..').resolve()
sys.path.append(str(project_root))

# Import project modules

from src.counterfactuals.core import (
    BasisGenerator, 
    TSFeatureSchema, 
    TargetSpec, 
    GeneratorConfig, 
    LossWeights,
)
from src.counterfactuals.basis import BSplineBasis


from torch.utils.data import DataLoader

from src.data_loader.wind_turbine.turbine_data_loader import get_turbine_data_loader
from src.models.wind_turbine.anomaly_transformer.model.AnomalyTransformer import AnomalyTransformer

# Device config
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device = torch.device("cpu")
print(f"Using device: {device}")

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

Using device: cpu


In [2]:
device = torch.device("cpu")
print(f"Using device: {device}")

Using device: cpu


In [3]:
# PROCESSED_DIR = project_root / "data" / "processed" / "ieee_phm"
DATA_DIR = project_root / "data" / "raw" / "wind_turbine"
DATA_DIR_TRAIN = DATA_DIR / "train"
DATA_DIR_TEST = DATA_DIR / "test"

OUTPUT_DIR = project_root / "outputs" / "wind_turbine"
SAVE_MODEL_DIR = OUTPUT_DIR / "saved_models"

# print(f"Processed data directory: {PROCESSED_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Model save directory: {SAVE_MODEL_DIR}")

Output directory: /home/rdb/Documents/nirban_documents/python_programs/counterfactual_basis_kernel/outputs/wind_turbine
Model save directory: /home/rdb/Documents/nirban_documents/python_programs/counterfactual_basis_kernel/outputs/wind_turbine/saved_models


In [4]:
# Configuration
WIN_SIZE = 144
STEP = 144
BATCH_SIZE = 64
INPUT_C = 82
OUTPUT_C = 82
E_LAYERS = 3
TEMPERATURE = 50
ANORMLY_RATIO = 1.5

# Model class mapping
MODEL_CLASSES = {
    '0': 'Generator Bearing',
    '1': 'Gearbox',
    '2': 'Transformer',
    '3': 'Hydraulic'
}

# Threshold map (from solver.py)
THRESH_MAP = {
    '0': 0.00013377491304709106,
    '1': 0.00044517662157886675,
    '2': 0.00014006024604896057,
    '3': 7.12080005177995e-05
}

DATASET_NAME = "Wind Farm A"

In [5]:
def my_kl_loss(p, q):
    res = p * (torch.log(p + 0.0001) - torch.log(q + 0.0001))
    return torch.mean(torch.sum(res, dim=-1), dim=1)

In [9]:
# filepath: /home/rdb/Documents/nirban_documents/python_programs/counterfactual_basis_kernel/notebooks/wind_turbine/01_load_data_and_models.ipynb
import joblib

# ===== SELECT WHICH MODEL CLASS TO ANALYZE =====
MODEL_ = '0'  # Change to '0', '1', '2', or '3'

print(f"Loading model class: {MODEL_} ({MODEL_CLASSES[MODEL_]})")

model_name = DATASET_NAME + MODEL_

# Paths
scaler_path = OUTPUT_DIR / f"model_{MODEL_}" / "scaler.joblib"
checkpoint_path = SAVE_MODEL_DIR / f"{DATASET_NAME}{MODEL_}_checkpoint1_.pth"

# Load scaler
scaler = joblib.load(scaler_path)
print(f"Scaler loaded from {scaler_path}")

# Load test data
test_loader, features, _, _ = get_turbine_data_loader(
    str(DATA_DIR_TEST), model_name,
    batch_size=BATCH_SIZE, win_size=WIN_SIZE, step=STEP,
    mode='test', scaler=scaler
)

# Load train data (for threshold calibration reference)
train_loader, _, _, background_data = get_turbine_data_loader(
    str(DATA_DIR_TRAIN), model_name,
    batch_size=BATCH_SIZE, win_size=WIN_SIZE, step=STEP,
    mode='train', scaler=scaler
)

# Load validation data
vali_loader, _, _, _ = get_turbine_data_loader(
    str(DATA_DIR_TRAIN), model_name,
    batch_size=BATCH_SIZE, win_size=WIN_SIZE, step=STEP,
    mode='val', scaler=scaler
)

print(f"\nNumber of features: {len(features)}")
print(f"Features: {features[:10]}... (showing first 10)")

Loading model class: 0 (Generator Bearing)
Scaler loaded from /home/rdb/Documents/nirban_documents/python_programs/counterfactual_basis_kernel/outputs/wind_turbine/model_0/scaler.joblib
test: torch.Size([7777, 82])


/home/rdb/Documents/nirban_documents/python_programs/.venv/lib/python3.12/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.3.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


train: torch.Size([46800, 82])
val: torch.Size([5328, 82])

Number of features: 82
Features: ['status_type_id', 'sensor_0_avg', 'sensor_1_avg', 'sensor_2_avg', 'wind_speed_3_avg', 'wind_speed_4_avg', 'wind_speed_3_max', 'wind_speed_3_min', 'wind_speed_3_std', 'sensor_5_avg']... (showing first 10)


In [7]:
# Build and load model
model = AnomalyTransformer(
    win_size=WIN_SIZE,
    enc_in=INPUT_C,
    c_out=OUTPUT_C,
    e_layers=E_LAYERS
)

model.load_state_dict(torch.load(checkpoint_path, map_location=device), strict=False)
model.to(device)
model.eval()

thresh = THRESH_MAP[MODEL_]
print(f"Model loaded from: {checkpoint_path}")
print(f"Threshold for class {MODEL_} ({MODEL_CLASSES[MODEL_]}): {thresh:.10f}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

Model loaded from: /home/rdb/Documents/nirban_documents/python_programs/counterfactual_basis_kernel/outputs/wind_turbine/saved_models/Wind Farm A0_checkpoint1_.pth
Threshold for class 0 (Generator Bearing): 0.0001337749
Model parameters: 4,918,389


# Generate Counterfactuals

In [10]:
# ── Fix 1: Per-Timestep Anomaly Wrapper ──
# The original wrapper uses MEAN score, but the detector uses PER-TIMESTEP thresholding.
# We need a differentiable proxy that targets per-timestep scores.

class AnomalyTransformerWrapperV2(torch.nn.Module):
    """
    V2: Returns a scalar that is a SOFT COUNT of how many timesteps exceed threshold.
    This directly optimizes what the anomaly detector actually checks.
    
    Uses: sigmoid((score_t - thresh) / tau) summed over timesteps.
    Minimizing this → fewer timesteps above threshold → classified as normal.
    """
    def __init__(self, model, win_size=144, temperature=50, 
                 thresh=1e-4, sigmoid_tau=None):
        super().__init__()
        self.model = model
        self.win_size = win_size
        self.temperature = temperature
        self.thresh = thresh
        # tau controls sigmoid sharpness: smaller = sharper but harder to optimize
        # Auto-scale based on threshold magnitude
        self.sigmoid_tau = sigmoid_tau if sigmoid_tau else max(thresh * 0.5, 1e-6)
        
    def forward(self, x):
        """
        x: (N, T, D) input windows
        Returns: (N, 1) — soft count of anomalous timesteps (minimize to 0)
        """
        output, series, prior, sigmas = self.model(x)
        
        # Reconstruction loss per timestep
        criterion = torch.nn.MSELoss(reduction='none')
        loss = torch.mean(criterion(x, output), dim=-1)  # (N, T)
        
        # Association discrepancy
        series_loss = 0.0
        prior_loss = 0.0
        for u in range(len(prior)):
            norm_prior = prior[u] / torch.unsqueeze(
                torch.sum(prior[u], dim=-1), dim=-1
            ).repeat(1, 1, 1, self.win_size)
            
            if u == 0:
                series_loss = self._kl_loss(series[u], norm_prior.detach()) * self.temperature
                prior_loss = self._kl_loss(norm_prior, series[u].detach()) * self.temperature
            else:
                series_loss += self._kl_loss(series[u], norm_prior.detach()) * self.temperature
                prior_loss += self._kl_loss(norm_prior, series[u].detach()) * self.temperature
        
        metric = torch.softmax((-series_loss - prior_loss), dim=-1)  # (N, T)
        per_timestep_score = metric * loss  # (N, T)
        
        # Soft count of timesteps above threshold
        # sigmoid((score - thresh) / tau): ≈1 if above thresh, ≈0 if below
        soft_above = torch.sigmoid(
            (per_timestep_score - self.thresh) / self.sigmoid_tau
        )  # (N, T)
        
        # Normalized: fraction of timesteps above threshold
        soft_anomaly_fraction = soft_above.mean(dim=1, keepdim=True)  # (N, 1)
        
        return soft_anomaly_fraction
    
    @staticmethod
    def _kl_loss(p, q):
        res = p * (torch.log(p + 0.0001) - torch.log(q + 0.0001))
        return torch.mean(torch.sum(res, dim=-1), dim=1)


# Also keep a max-score wrapper as alternative
class AnomalyTransformerWrapperMax(torch.nn.Module):
    """
    V3: Returns the MAX per-timestep anomaly score.
    If max < thresh, ALL timesteps are below threshold → definitely normal.
    Sharper signal but may be harder to optimize.
    Uses soft-max (LogSumExp) for differentiability.
    """
    def __init__(self, model, win_size=144, temperature=50, softmax_temp=50.0):
        super().__init__()
        self.model = model
        self.win_size = win_size
        self.temperature = temperature
        self.softmax_temp = softmax_temp  # higher = closer to hard max
        
    def forward(self, x):
        output, series, prior, sigmas = self.model(x)
        criterion = torch.nn.MSELoss(reduction='none')
        loss = torch.mean(criterion(x, output), dim=-1)
        
        series_loss = 0.0
        prior_loss = 0.0
        for u in range(len(prior)):
            norm_prior = prior[u] / torch.unsqueeze(
                torch.sum(prior[u], dim=-1), dim=-1
            ).repeat(1, 1, 1, self.win_size)
            if u == 0:
                series_loss = self._kl_loss(series[u], norm_prior.detach()) * self.temperature
                prior_loss = self._kl_loss(norm_prior, series[u].detach()) * self.temperature
            else:
                series_loss += self._kl_loss(series[u], norm_prior.detach()) * self.temperature
                prior_loss += self._kl_loss(norm_prior, series[u].detach()) * self.temperature
        
        metric = torch.softmax((-series_loss - prior_loss), dim=-1)
        per_timestep_score = metric * loss  # (N, T)
        
        # Smooth max via LogSumExp
        smooth_max = (1.0 / self.softmax_temp) * torch.logsumexp(
            self.softmax_temp * per_timestep_score, dim=1, keepdim=True
        )  # (N, 1)
        
        return smooth_max
    
    @staticmethod
    def _kl_loss(p, q):
        res = p * (torch.log(p + 0.0001) - torch.log(q + 0.0001))
        return torch.mean(torch.sum(res, dim=-1), dim=1)


# Build both wrappers
wrapped_model_v2 = AnomalyTransformerWrapperV2(
    model, win_size=WIN_SIZE, temperature=TEMPERATURE,
    thresh=thresh, sigmoid_tau=thresh * 0.3
)
wrapped_model_v2.to(device).eval()

wrapped_model_max = AnomalyTransformerWrapperMax(
    model, win_size=WIN_SIZE, temperature=TEMPERATURE,
    softmax_temp=1.0 / (thresh * 2)  # scale so scores near thresh give meaningful gradients
)
wrapped_model_max.to(device).eval()

# Quick test
with torch.no_grad():
    test_batch = next(iter(test_loader))
    test_inp = test_batch[0][:4].float().to(device)
    scores_v2 = wrapped_model_v2(test_inp).squeeze()
    scores_max = wrapped_model_max(test_inp).squeeze()
    print(f"V2 (soft fraction above thresh): {scores_v2.tolist()}")
    print(f"Max wrapper scores:              {scores_max.tolist()}")
    print(f"Threshold:                       {thresh:.10f}")

print("\n✅ V2 and Max wrappers ready")

V2 (soft fraction above thresh): [0.1832936853170395, 0.18536384403705597, 0.242144376039505, 0.3604549169540405]
Max wrapper scores:              [0.001395826111547649, 0.0013957732589915395, 0.0014643429312855005, 0.001628477475605905]
Threshold:                       0.0001337749

✅ V2 and Max wrappers ready


In [14]:
# ── Collect anomalous windows from test set ──
# We need: (1) anomalous input windows, (2) their timestamps, (3) scores

criterion = torch.nn.MSELoss(reduction='none')

anomalous_windows = []   # list of (input_tensor, score, timestamps, window_idx)
normal_windows = []
all_window_scores = []

with torch.no_grad():
    window_idx = 0
    for i, (input_data, labels, date_time) in enumerate(test_loader):
        inp = input_data.float().to(device)
        scores = wrapped_model(inp).squeeze(-1)  # (B,)
        
        # Also compute per-timestep scores for detailed analysis
        output, series, prior, sigmas = model(inp)
        loss = torch.mean(criterion(inp, output), dim=-1)
        
        series_loss = 0.0
        prior_loss = 0.0
        for u in range(len(prior)):
            norm_prior = prior[u] / torch.unsqueeze(
                torch.sum(prior[u], dim=-1), dim=-1
            ).repeat(1, 1, 1, WIN_SIZE)
            if u == 0:
                series_loss = my_kl_loss(series[u], norm_prior.detach()) * TEMPERATURE
                prior_loss = my_kl_loss(norm_prior, series[u].detach()) * TEMPERATURE
            else:
                series_loss += my_kl_loss(series[u], norm_prior.detach()) * TEMPERATURE
                prior_loss += my_kl_loss(norm_prior, series[u].detach()) * TEMPERATURE
        
        metric = torch.softmax((-series_loss - prior_loss), dim=-1)
        cri = (metric * loss).detach().cpu().numpy()
        
        date_np = np.array(date_time)
        
        for b in range(inp.shape[0]):
            window_score = scores[b].item()
            true_label = labels[b].item() if labels.ndim > 0 else labels.item()
            timestep_scores = cri[b]
            
            # Check if anomalous using the same criterion as the original code
            is_anomalous = int(np.sum(timestep_scores > thresh) >= np.round(ANORMLY_RATIO * timestep_scores.shape[0] / 100))
            
            ts = [str(date_np[b][t][0]) if hasattr(date_np[b][t], '__len__') else str(date_np[b][t]) 
                  for t in range(len(date_np[b]))]
            
            record = {
                'input': inp[b].cpu(),
                'window_score': window_score,
                'timestep_scores': timestep_scores,
                'timestamps': ts,
                'window_idx': window_idx,
                'true_label': int(true_label),
                'pred_anomalous': is_anomalous,
            }
            
            all_window_scores.append(window_score)
            
            if is_anomalous:
                anomalous_windows.append(record)
            else:
                normal_windows.append(record)
            
            window_idx += 1

print(f"Total windows: {window_idx}")
print(f"Anomalous windows: {len(anomalous_windows)}")
print(f"Normal windows: {len(normal_windows)}")

# Sort anomalous windows by score (highest first)
anomalous_windows.sort(key=lambda r: r['window_score'], reverse=True)

if len(anomalous_windows) > 0:
    print(f"\nTop-5 anomalous window scores:")
    for j in range(min(5, len(anomalous_windows))):
        w = anomalous_windows[j]
        print(f"  Window {w['window_idx']}: score={w['window_score']:.8f}, "
              f"true_label={w['true_label']}, ts={w['timestamps'][0]}..{w['timestamps'][-1]}")

NameError: name 'wrapped_model' is not defined

In [11]:
# ── Fix 2: Adjusted targets and weights for V2 wrapper ──
# 
# V2 wrapper outputs a FRACTION (0..1) of timesteps above threshold.
# The anomaly criterion is: fraction >= ANORMLY_RATIO / 100 = 0.015
# So we need the fraction to be < 0.015.
# Target: 0.0 (no timesteps above threshold)

anomaly_fraction_cutoff = ANORMLY_RATIO / 100.0  # 0.015
print(f"Anomaly fraction cutoff: {anomaly_fraction_cutoff}")

target_v2 = TargetSpec(
    task_type="regression",
    target_value=0.0,
    target_range=(0.0, anomaly_fraction_cutoff * 0.5),  # safely below cutoff
)

# For max wrapper: push max score below threshold
target_max = TargetSpec(
    task_type="regression",
    target_value=thresh * 0.3,  # well below threshold
    target_range=(0.0, thresh * 0.8),
)

# Adjusted loss weights — the V2 output is in [0,1] range, much larger gradients
loss_weights_v2 = LossWeights(
    validity=20.0,       # Strong push — output is now in interpretable range
    proximity=5.0,       # Slightly relaxed — allow more change for validity
    sparsity=0.2,
    diversity=0.0,
    smoothness=1.5,
    channel_sparsity=0.0,
    state_lock=0.0,
)

# More aggressive config for harder anomalies
gen_config_v2 = GeneratorConfig(
    editable_roles=("action",),
    allow_state_edits=False,
    init_std=0.01,          # slightly larger init
    lr=0.01,                # higher LR — the loss surface is smoother now
    max_iter=3000,          # more iterations
    eta_min=1e-5,
    gradient_clip_norm=2.0, # allow larger gradients
    early_stop_tol_reg=anomaly_fraction_cutoff * 0.5,  # stop when fraction < half cutoff
    early_stop_patience=500,
    clamp_during_optim=True,
)

print(f"✅ V2 targets and config defined")
print(f"   Target: fraction < {anomaly_fraction_cutoff * 0.5:.4f}")
print(f"   LR: {gen_config_v2.lr}, max_iter: {gen_config_v2.max_iter}")

Anomaly fraction cutoff: 0.015
✅ V2 targets and config defined
   Target: fraction < 0.0075
   LR: 0.01, max_iter: 3000


In [12]:
# ── Fix 3: Rebuild generators with V2 wrapper ──

basis_types = ['fourier', 'bspline', 'wavelet', 'polynomial', 'rbf']
generators_v2 = {}

for btype in basis_types:
    generators_v2[btype] = BasisGenerator(
        model=wrapped_model_v2,       # ← V2 wrapper with per-timestep targeting
        sequence_length=WIN_SIZE,
        feature_dim=INPUT_C,
        basis_type=btype,
        num_basis=12,
        device=str(device),
        config=gen_config_v2,
    )

print(f"✅ {len(generators_v2)} V2 generators initialized")

✅ 5 V2 generators initialized


In [13]:
# ── Fix 4: Generate CFs with V2 wrapper ──

N_WINDOWS = min(10, len(anomalous_windows))
NUM_CFS = 3

print(f"Generating CFs with V2 wrapper for {N_WINDOWS} anomalous windows × {len(generators_v2)} generators")

all_cf_results_v2 = {}

def check_cf_validity(cf_tensor, raw_model, thresh, win_size, anormly_ratio, temperature):
    """Check if a CF window passes the actual anomaly detection criterion."""
    with torch.no_grad():
        output_cf, series_cf, prior_cf, _ = raw_model(cf_tensor)
        loss_cf = torch.mean(torch.nn.MSELoss(reduction='none')(cf_tensor, output_cf), dim=-1)
        
        s_loss, p_loss = 0.0, 0.0
        for u in range(len(prior_cf)):
            np_cf = prior_cf[u] / torch.unsqueeze(
                torch.sum(prior_cf[u], dim=-1), dim=-1
            ).repeat(1, 1, 1, win_size)
            if u == 0:
                s_loss = my_kl_loss(series_cf[u], np_cf.detach()) * temperature
                p_loss = my_kl_loss(np_cf, series_cf[u].detach()) * temperature
            else:
                s_loss += my_kl_loss(series_cf[u], np_cf.detach()) * temperature
                p_loss += my_kl_loss(np_cf, series_cf[u].detach()) * temperature
        
        met_cf = torch.softmax((-s_loss - p_loss), dim=-1)
        cri_cf = (met_cf * loss_cf).detach().cpu().numpy()[0]
    
    is_normal = int(np.sum(cri_cf > thresh) < np.round(anormly_ratio * cri_cf.shape[0] / 100))
    return is_normal, cri_cf


for gen_name, gen in generators_v2.items():
    print(f"\n{'='*70}")
    print(f"  Generator: {gen_name.upper()} (V2)")
    print(f"{'='*70}")
    
    gen_results = []
    
    for w_idx in range(N_WINDOWS):
        window = anomalous_windows[w_idx]
        query = window['input'].float()
        
        try:
            original_score = window['window_score']
            
            # V2 score for reference
            with torch.no_grad():
                orig_v2_score = wrapped_model_v2(query.unsqueeze(0).to(device)).item()
            
            cfs, info = gen.generate(
                query_instance=query,
                target=target_v2,
                schema=wind_schema,
                num_cfs=NUM_CFS,
                loss_weights=loss_weights_v2,
                verbose=False,
            )
            
            # Evaluate ALL CFs with the actual anomaly criterion (not just wrapper score)
            best_cf = None
            best_cf_score = float('inf')
            best_cf_is_normal = False
            best_cri_cf = None
            
            for cf_idx in range(cfs.shape[0]):
                cf_tensor = cfs[cf_idx:cf_idx+1].float().to(device)
                is_normal, cri_cf = check_cf_validity(
                    cf_tensor, model, thresh, WIN_SIZE, ANORMLY_RATIO, TEMPERATURE
                )
                
                # Also get the mean score for ranking
                with torch.no_grad():
                    cf_mean_score = wrapped_model(cf_tensor).item()
                
                # Prefer: (1) valid CFs, (2) lowest mean score
                if is_normal and not best_cf_is_normal:
                    best_cf = cfs[cf_idx].detach().cpu().numpy()
                    best_cf_score = cf_mean_score
                    best_cf_is_normal = True
                    best_cri_cf = cri_cf
                elif (is_normal == best_cf_is_normal) and cf_mean_score < best_cf_score:
                    best_cf = cfs[cf_idx].detach().cpu().numpy()
                    best_cf_score = cf_mean_score
                    best_cf_is_normal = is_normal
                    best_cri_cf = cri_cf
            
            if best_cf is None:
                best_cf = cfs[0].detach().cpu().numpy()
                best_cf_score = 0.0
                best_cri_cf = np.zeros(WIN_SIZE)
            
            # Statistics
            delta = np.abs(best_cf - query.numpy())
            mean_delta = delta.mean()
            l2_dist = np.sqrt(np.sum((best_cf - query.numpy()) ** 2))
            feature_changes = np.mean(delta, axis=0)
            sparsity = np.mean(feature_changes < 0.01)
            
            # Count how many timesteps are above threshold
            n_above = np.sum(best_cri_cf > thresh)
            n_needed = np.round(ANORMLY_RATIO * WIN_SIZE / 100)
            
            gen_results.append({
                'window_global_idx': window['window_idx'],
                'true_label': window['true_label'],
                'original_score': original_score,
                'cf_score': best_cf_score,
                'cf_timestep_scores': best_cri_cf,
                'cf_is_normal': best_cf_is_normal,
                'n_timesteps_above_thresh': n_above,
                'n_needed_for_anomaly': n_needed,
                'original': query.numpy(),
                'cf': best_cf,
                'timestamps': window['timestamps'],
                'mean_delta': mean_delta,
                'l2_distance': l2_dist,
                'sparsity': sparsity,
                'feature_changes': feature_changes,
                'info': info,
            })
            
            status = "✅ VALID" if best_cf_is_normal else f"❌ ({int(n_above)}/{int(n_needed)} ts above)"
            print(f"  Window {window['window_idx']:3d}: "
                  f"orig={original_score:.8f} → cf={best_cf_score:.8f} | "
                  f"Δmean={mean_delta:.4f} | v2={orig_v2_score:.4f}→{info.get('final_validity_err', '?')} | "
                  f"{status}")
            
        except Exception as e:
            import traceback
            print(f"  ⚠️ Window {window['window_idx']} failed: {e}")
            traceback.print_exc()
    
    all_cf_results_v2[gen_name] = gen_results
    
    valid_count = sum(1 for r in gen_results if r['cf_is_normal'])
    print(f"\n  Summary: {valid_count}/{len(gen_results)} valid CFs "
          f"({100*valid_count/max(len(gen_results),1):.1f}%)")

print(f"\n{'='*70}")
print("All V2 generators finished.")

NameError: name 'anomalous_windows' is not defined